# 3일차 실습: 실시간 웹캠 필터 앱
---
## 오늘 목표
1~2일차에서 배운 모든 기술을 합쳐서 완성도 높은 앱을 만듭니다.

### 필터 앱 완성
- Step 1: 웹캠 연결 확인
- Step 2: 필터 함수 만들기 (6가지)
- Step 3: 메인 앱 실행
- Step 4: 트랙바로 파라미터 실시간 조절
- Step 5: 웹캠 영상 녹화 기능 추가

### 심화 실습
- Step 6: Day2 감정/포즈 파이프라인 실시간 연결
- Step 7: Streamlit 웹 앱 배포

### 키보드 조작
| 키 | 기능 |
|---|---|
| `1~6` | 필터 전환 |
| `r` | 녹화 시작/중지 |
| `s` | 현재 화면 저장 |
| `q` | 종료 |

## 0. 환경 설정 — OpenVINO 로컬 설치

처음 수업 시작 전에 한 번만 실행하면 됩니다.

### Python 3.11 설치 (3.13이면 필수)

**Windows**
1. https://www.python.org/downloads/release/python-3119/ 에서 다운로드
2. 설치 시 **'Add to PATH'** 체크 필수!

**Mac**
```bash
brew install python@3.11
```

### 가상환경 생성 및 패키지 설치

**Windows (터미널)**
```bash
py -3.11 -m venv cv_env
.\venv\Scripts\activate
pip install openvino openvino-dev opencv-python numpy matplotlib
```

**Mac (터미널)**
```bash
python3.11 -m venv cv_env
source cv_env/bin/activate
pip install openvino openvino-dev opencv-python numpy matplotlib
```

### VS Code에서 가상환경 선택
1. 우측 상단 '커널 선택' 클릭
2. '다른 커널 선택'
3. 'Python 환경'
4. '+ Python 환경 만들기'
5. 'venv' 클릭
6. 폴더 선택
7. '사용자 지정'
8. 3.11 버전 선택



##### 터미널 열어서 실행
```
pip install opencv-python openvino openvino-dev numpy matplotlib pillow
```


In [2]:
# 설치 확인 — 아래 셀을 실행해서 모두 OK 뜨면 준비 완료!
import sys
print(f'Python 버전: {sys.version}')

try:
    import cv2
    print(f'OpenCV: {cv2.__version__}  OK')
except ImportError:
    print('OpenCV 미설치 — pip install opencv-python')

try:
    from openvino.runtime import Core
    ie = Core()
    print(f'OpenVINO OK  사용 가능한 디바이스: {ie.available_devices}')
except ImportError:
    print('OpenVINO 미설치 — pip install openvino openvino-dev')

try:
    import numpy as np
    print(f'NumPy: {np.__version__}  OK')
except ImportError:
    print('NumPy 미설치 — pip install numpy')

try:
    import matplotlib
    print(f'Matplotlib: {matplotlib.__version__}  OK')
except ImportError:
    print('Matplotlib 미설치 — pip install matplotlib')


Python 버전: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
OpenCV: 4.11.0  OK


: 

In [1]:
from PIL import ImageFont, ImageDraw, Image
import numpy as np
import cv2

# 한글 폰트 로드 (Windows 기본 폰트)
font_path = "C:/Windows/Fonts/malgun.ttf"   # 맑은 고딕
font = ImageFont.truetype(font_path, 24)     # 크기 24

def put_korean_text(img, text, position, color=(0, 255, 255)):
    """OpenCV 이미지에 한글 텍스트 쓰기"""
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    draw.text(position, text, font=font, fill=color)
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

## Step 1 — 웹캠 연결 확인

In [2]:
import cv2
import numpy as np
from datetime import datetime

camera = cv2.VideoCapture(0)
if not camera.isOpened():
    print('❌ 웹캠을 열 수 없습니다.')
else:
    ret, frame = camera.read()
    print(f'✅ 웹캠 연결 성공! 해상도: {frame.shape[1]}x{frame.shape[0]}')
    camera.release()

✅ 웹캠 연결 성공! 해상도: 640x480


## Step 2 — 필터 함수 만들기

각 필터를 함수로 분리해서 깔끔하게 관리합니다.

In [5]:
# 각 필터를 함수로 정의

def apply_original(frame):
    return frame                                              # 원본 그대로

def apply_grayscale(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)            # 회색조 → 다시 BGR

def apply_blur(frame):
    return cv2.GaussianBlur(frame, (21, 21), 100)             # Gaussian 블러

def apply_canny(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)             # 노이즈 제거
    edges = cv2.Canny(blurred, 50, 150)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

def apply_contour(frame):
    result = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    large = [c for c in contours if cv2.contourArea(c) > 1000]
    cv2.drawContours(result, large, -1, (0, 255, 0), 2)
    for c in large:
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(result, (x, y), (x+w, y+h), (0, 0, 255), 1)
    cv2.putText(result, f'Objects: {len(large)}', (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return result

def apply_color_mask(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([100, 50, 50]), np.array([130, 255, 255]))
    return cv2.bitwise_and(frame, frame, mask=mask)

# 필터 딕셔너리: 키코드 → (함수, 이름)
FILTERS = {
    ord('1'): (apply_original,   '1: 원본'),
    ord('2'): (apply_grayscale,  '2: 회색조'),
    ord('3'): (apply_blur,       '3: 블러'),
    ord('4'): (apply_canny,      '4: Canny 엣지'),
    ord('5'): (apply_contour,    '5: Contour'),
    ord('6'): (apply_color_mask, '6: 파란색 마스크'),
}
print('✅ 필터 함수 정의 완료!')
print('사용 가능한 필터:', [v[1] for v in FILTERS.values()])

✅ 필터 함수 정의 완료!
사용 가능한 필터: ['1: 원본', '2: 회색조', '3: 블러', '4: Canny 엣지', '5: Contour', '6: 파란색 마스크']


## Step 3 — 메인 앱 실행

모든 필터를 합쳐서 인터랙티브 앱을 실행합니다.

In [6]:
# 실시간 웹캠 필터 앱
camera = cv2.VideoCapture(0)
current_filter_func = apply_original
current_filter_name = '1: 원본'
saved_count = 0

print('앱 시작! 키 조작: 1~6 필터 전환 / s 저장 / q 종료')

while True:
    ret, frame = camera.read()
    if not ret:
        break

    filtered = current_filter_func(frame)                       # 필터 적용

    # 정보 오버레이
    timestamp = datetime.now().strftime('%H:%M:%S')
    cv2.putText(filtered, f'Filter: {current_filter_name}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(filtered, timestamp, (10, filtered.shape[0] - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    cv2.putText(filtered, 's:저장  q:종료', (filtered.shape[1]-170, filtered.shape[0]-15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

    cv2.imshow('실시간 필터 앱 (1~6: 필터 전환)', filtered)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('s'):
        filename = f"capture_{datetime.now().strftime('%H%M%S')}.png"
        cv2.imwrite(filename, filtered)
        saved_count += 1
        print(f'저장됨: {filename} (총 {saved_count}장)')
    elif key in FILTERS:
        current_filter_func, current_filter_name = FILTERS[key]
        print(f'필터 변경: {current_filter_name}')

camera.release()
cv2.destroyAllWindows()
print(f'앱 종료. 총 {saved_count}장 저장됨.')

앱 시작! 키 조작: 1~6 필터 전환 / s 저장 / q 종료
앱 종료. 총 0장 저장됨.


## Step 4 — 트랙바로 파라미터 실시간 조절

`cv2.createTrackbar()`를 사용하면 슬라이더로 값을 바꾸면서 결과를 실시간으로 확인할 수 있습니다.

Canny 엣지의 threshold 값을 트랙바로 조절해봅니다.

> 💡 트랙바는 `cv2.namedWindow()`로 만든 창에만 붙일 수 있어요.

In [7]:
# 트랙바로 Canny threshold 실시간 조절
camera = cv2.VideoCapture(0)

# 트랙바 창 생성
cv2.namedWindow('Canny 트랙바 (q: 종료)')
cv2.createTrackbar('Threshold1', 'Canny 트랙바 (q: 종료)', 50,  300, lambda x: None)
cv2.createTrackbar('Threshold2', 'Canny 트랙바 (q: 종료)', 150, 600, lambda x: None)
cv2.createTrackbar('Blur',       'Canny 트랙바 (q: 종료)', 5,   21,  lambda x: None)

while True:
    ret, frame = camera.read()
    if not ret:
        break

    # 트랙바 값 읽기
    t1   = cv2.getTrackbarPos('Threshold1', 'Canny 트랙바 (q: 종료)')
    t2   = cv2.getTrackbarPos('Threshold2', 'Canny 트랙바 (q: 종료)')
    blur = cv2.getTrackbarPos('Blur',       'Canny 트랙바 (q: 종료)')

    # blur 값은 홀수여야 함
    blur = blur if blur % 2 == 1 else blur + 1
    blur = max(1, blur)

    # Canny 적용
    gray    = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (blur, blur), 0)
    edges   = cv2.Canny(blurred, t1, t2)
    edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    # 파라미터 값 화면에 표시
    cv2.putText(edges_bgr, f'T1:{t1}  T2:{t2}  Blur:{blur}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    # 원본과 나란히 표시
    combined = np.hstack([frame, edges_bgr])
    cv2.imshow('Canny 트랙바 (q: 종료)', combined)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

camera.release()
cv2.destroyAllWindows()


### 🎯 GenAI 활용 — 트랙바 확장

---
**프롬프트:**
```
Python OpenCV 트랙바 앱에 다음을 추가해줘:

현재 Canny 엣지에 트랙바가 있는데,
트랙바로 선택 가능한 필터 모드를 추가해줘:
- Mode 0: Canny 엣지
- Mode 1: Gaussian 블러 (커널 크기 트랙바)
- Mode 2: HSV 마스크 (H 범위 트랙바 2개)

'Mode' 트랙바 값에 따라 다른 필터가 적용되고
현재 모드 이름이 화면 상단에 표시되도록 해줘.
```
---

In [8]:
import cv2
import numpy as np

# 트랙바 콜백용 빈 함수
def nothing(x):
    pass

# 카메라 설정
camera = cv2.VideoCapture(0)
window_name = 'Multi-Filter App (q: Quit)'
cv2.namedWindow(window_name)

# 1. 트랙바 생성
cv2.createTrackbar('Mode', window_name, 0, 2, nothing)       # 0:Canny, 1:Blur, 2:HSV
cv2.createTrackbar('T1', window_name, 50, 300, nothing)      # Canny Threshold 1
cv2.createTrackbar('T2', window_name, 150, 600, nothing)     # Canny Threshold 2
cv2.createTrackbar('Blur', window_name, 5, 21, nothing)      # Blur 커널 크기
cv2.createTrackbar('H_Low', window_name, 0, 179, nothing)    # HSV 하한값
cv2.createTrackbar('H_High', window_name, 179, 179, nothing) # HSV 상한값

while True:
    ret, frame = camera.read()
    if not ret:
        break

    # 2. 트랙바 실시간 값 읽기
    mode = cv2.getTrackbarPos('Mode', window_name)
    t1 = cv2.getTrackbarPos('T1', window_name)
    t2 = cv2.getTrackbarPos('T2', window_name)
    blur_val = cv2.getTrackbarPos('Blur', window_name)
    h_low = cv2.getTrackbarPos('H_Low', window_name)
    h_high = cv2.getTrackbarPos('H_High', window_name)

    # 공통 처리: Blur 값 홀수 보정
    blur_val = blur_val if blur_val % 2 == 1 else blur_val + 1
    blur_val = max(1, blur_val)

    # 3. 모드별 필터 적용
    result_frame = frame.copy()
    mode_text = ""

    # --- Mode 0: Canny Edge ---
    if mode == 0:
        mode_text = "MODE: Canny Edge"
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (blur_val, blur_val), 0)
        edges = cv2.Canny(blurred, t1, t2)
        result_frame = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    # --- Mode 1: Gaussian Blur ---
    elif mode == 1:
        mode_text = f"MODE: Gaussian Blur (Size:{blur_val})"
        result_frame = cv2.GaussianBlur(frame, (blur_val, blur_val), 0)

    # --- Mode 2: HSV Mask ---
    elif mode == 2:
        mode_text = f"MODE: HSV Mask (H:{h_low}~{h_high})"
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        # S(채도)와 V(명도)는 기본 범위를 넓게 잡아 색상만 추출하도록 설정
        lower_hsv = np.array([h_low, 50, 50])
        upper_hsv = np.array([h_high, 255, 255])
        
        mask = cv2.inRange(hsv, lower_hsv, upper_hsv)
        result_frame = cv2.bitwise_and(frame, frame, mask=mask)

    # 4. 정보 표시 (모드 이름 및 파라미터)
    cv2.putText(result_frame, mode_text, (20, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # 결과 출력 (원본과 필터 결과 나란히 보기)
    combined = np.hstack([frame, result_frame])
    cv2.imshow(window_name, combined)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

camera.release()
cv2.destroyAllWindows()

## Step 5 — 웹캠 영상 녹화 기능 추가

`cv2.VideoWriter`로 필터가 적용된 영상을 파일로 저장할 수 있습니다.

```
r 키 → 녹화 시작  →  파일에 프레임 저장  →  r 키 → 녹화 중지
```

In [9]:
# 녹화 기능이 추가된 웹캠 필터 앱
from datetime import datetime

camera  = cv2.VideoCapture(0)
ret, frame = camera.read()
h, w = frame.shape[:2]

# VideoWriter 설정
fourcc    = cv2.VideoWriter_fourcc(*'mp4v')  # mp4 포맷
writer    = None
recording = False

current_filter_func = apply_original
current_filter_name = '1: 원본'
saved_count = 0

print('앱 시작!  r: 녹화시작/중지  s: 캡처  q: 종료')

while True:
    ret, frame = camera.read()
    if not ret:
        break

    filtered = current_filter_func(frame)

    # 녹화 중이면 프레임 저장
    if recording and writer is not None:
        writer.write(filtered)

    # 정보 오버레이
    timestamp = datetime.now().strftime('%H:%M:%S')
    rec_text  = '● REC' if recording else ''
    cv2.putText(filtered, f'Filter: {current_filter_name}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(filtered, timestamp, (10, filtered.shape[0]-15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    if recording:
        cv2.putText(filtered, 'REC', (filtered.shape[1]-80, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

    cv2.imshow('필터 앱 with 녹화 (r:녹화  s:저장  q:종료)', filtered)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('r'):   # 녹화 시작/중지
        if not recording:
            fname   = f'recording_{datetime.now().strftime("%H%M%S")}.mp4'
            writer  = cv2.VideoWriter(fname, fourcc, 20.0, (w, h))
            recording = True
            print(f'녹화 시작: {fname}')
        else:
            writer.release()
            writer    = None
            recording = False
            print('녹화 중지')
    elif key == ord('s'):   # 화면 저장
        fname = f'capture_{datetime.now().strftime("%H%M%S")}.png'
        cv2.imwrite(fname, filtered)
        saved_count += 1
        print(f'저장: {fname}')
    elif key in FILTERS:
        current_filter_func, current_filter_name = FILTERS[key]

if writer:
    writer.release()
camera.release()
cv2.destroyAllWindows()
print(f'종료. 캡처 {saved_count}장')


앱 시작!  r: 녹화시작/중지  s: 캡처  q: 종료
종료. 캡처 0장


## 🎯 도전 과제 1 — GenAI 활용: 새 필터 추가

---
**프롬프트:**
```
Python OpenCV로 웹캠 영상에 적용할 새로운 필터 함수를 만들어줘.
다음 3가지 중 하나를 골라서 구현해줘:

A) 세피아 필터: 이미지를 빈티지 세피아 톤으로 변환 (numpy 행렬 변환 사용)
B) 카툰 필터: 만화처럼 보이게 변환 (bilateralFilter + Canny 엣지 합치기)
C) 야간 투시 필터: 초록색 야간 투시경 효과 (회색조를 초록 채널에만 적용)

함수 이름은 apply_새이름(frame) 형태로, frame 입력 받아 BGR 이미지 반환.
```
만든 함수를 FILTERS 딕셔너리에 ord('7') 키로 추가해서 테스트하세요!
---

In [10]:
import cv2
import numpy as np

# --- [1~8번 기존 필터 함수들] ---
def apply_sepia(frame):
    sepia_matrix = np.array([[0.131, 0.534, 0.272], [0.168, 0.686, 0.349], [0.189, 0.769, 0.393]])
    return np.clip(cv2.transform(frame, sepia_matrix), 0, 255).astype(np.uint8)

def apply_cartoon(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)
    edges = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 9)
    color = cv2.bilateralFilter(frame, 9, 250, 250)
    return cv2.bitwise_and(color, color, mask=edges)

def apply_night_vision(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    nv = np.zeros_like(frame); nv[:, :, 1] = gray
    return nv

def apply_sketch(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    inv = 255 - gray
    blur = cv2.GaussianBlur(inv, (21, 21), 0)
    sketch = cv2.divide(gray, 255 - blur, scale=256)
    return cv2.cvtColor(sketch, cv2.COLOR_GRAY2BGR)

def apply_pixelation(frame):
    h, w = frame.shape[:2]
    temp = cv2.resize(frame, (64, 64), interpolation=cv2.INTER_LINEAR)
    return cv2.resize(temp, (w, h), interpolation=cv2.INTER_NEAREST)

def apply_threshold(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    return cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)

def apply_retro_noise(frame):
    noise = np.zeros(frame.shape, np.uint8)
    cv2.randn(noise, (0,0,0), (50,50,50))
    return cv2.add(frame, noise)

def apply_oil_painting(frame):
    return cv2.stylization(frame, sigma_s=60, sigma_r=0.07)

# --- [새로 추가된 지능형 기능들] ---

# 얼굴 인식용 분류기 로드
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# 1. 움직임 감지용 변수
prev_frame = None

def func_motion_detector(frame):
    global prev_frame
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)
    
    if prev_frame is None:
        prev_frame = gray
        return frame
    
    # 프레임 차이 계산
    diff = cv2.absdiff(prev_frame, gray)
    _, thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
    prev_frame = gray
    
    # 차이가 있는 곳에 외곽선 그리기
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        if cv2.contourArea(c) < 500: continue
        (x, y, w, h) = cv2.boundingRect(c)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
    return frame

def func_face_detection(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(frame, 'Face', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
    return frame

def func_privacy_mask(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    for (x, y, w, h) in faces:
        face_roi = frame[y:y+h, x:x+w]
        # 얼굴 영역 모자이크 처리
        face_roi = cv2.resize(face_roi, (10, 10), interpolation=cv2.INTER_LINEAR)
        frame[y:y+h, x:x+w] = cv2.resize(face_roi, (w, h), interpolation=cv2.INTER_NEAREST)
    return frame

# 3. 사물 추적 (간단한 색상 기반 추적 예시: 초록색 대상)
def func_object_tracking(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    # 초록색 범위 (조정 가능)
    lower_green = np.array([35, 100, 100])
    upper_green = np.array([85, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        if cv2.contourArea(c) > 500:
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 3)
            cv2.putText(frame, "Tracking Green", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return frame

# --- [메인 실행 영역] ---

# 기능 매핑 (a, b, c, d 키 사용)
FUNCTIONS = {
    ord('a'): func_motion_detector,
    ord('s'): func_face_detection,
    ord('d'): func_object_tracking,
    ord('f'): func_privacy_mask
}

FILTERS = {
    ord('1'): apply_sepia, ord('2'): apply_cartoon, ord('3'): apply_night_vision,
    ord('4'): apply_sketch, ord('5'): apply_pixelation, ord('6'): apply_threshold,
    ord('7'): apply_retro_noise, ord('8'): apply_oil_painting
}

cap = cv2.VideoCapture(0)
active_mode = None
is_mirrored = True

print("="*50)
print(" [스마트 웹캠 통합 앱] ")
print(" 1~8: 필터 모드")
print(" a: 움직임 감지 | s: 얼굴 인식 | d: 색상 추적 | f: 모자이크")
print(" 0: 원본 | m: 거울 반전 | q: 종료")
print("="*50)

while True:
    ret, frame = cap.read()
    if not ret: break
    if is_mirrored: frame = cv2.flip(frame, 1)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): break
    elif key == ord('m'): is_mirrored = not is_mirrored
    elif key in FILTERS or key in FUNCTIONS: active_mode = key
    elif key == ord('0'): active_mode = None

    # 모드 적용
    mode_name = "ORIGINAL"
    if active_mode in FILTERS:
        frame = FILTERS[active_mode](frame)
        mode_name = FILTERS[active_mode].__name__.replace('apply_', '').upper()
    elif active_mode in FUNCTIONS:
        frame = FUNCTIONS[active_mode](frame)
        mode_name = FUNCTIONS[active_mode].__name__.replace('func_', '').upper()

    status = f"{mode_name} | MIRROR:{'ON' if is_mirrored else 'OFF'}"
    cv2.putText(frame, status, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.imshow('Smart WebCam Master', frame)

cap.release()
cv2.destroyAllWindows()

 [스마트 웹캠 통합 앱] 
 1~8: 필터 모드
 a: 움직임 감지 | s: 얼굴 인식 | d: 색상 추적 | f: 모자이크
 0: 원본 | m: 거울 반전 | q: 종료


## 🎯 도전 과제 2 — GenAI 활용: 화면 분할 비교

---
**프롬프트:**
```
Python OpenCV 웹캠 앱에서 화면을 좌우로 분할해서
왼쪽에는 원본, 오른쪽에는 Canny 엣지 결과를 동시에 표시하는 코드를 작성해줘.

조건:
- np.hstack()으로 두 이미지를 가로로 붙이기
- 각 화면 왼쪽 상단에 '원본', 'Canny 엣지' 텍스트 표시
- 'q' 키로 종료
```
---

In [11]:
import cv2
import numpy as np

# --- [1~8번 기존 필터 함수들] ---
def apply_sepia(frame):
    sepia_matrix = np.array([[0.131, 0.534, 0.272], [0.168, 0.686, 0.349], [0.189, 0.769, 0.393]])
    return np.clip(cv2.transform(frame, sepia_matrix), 0, 255).astype(np.uint8)

def apply_cartoon(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)
    edges = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 9)
    color = cv2.bilateralFilter(frame, 9, 250, 250)
    return cv2.bitwise_and(color, color, mask=edges)

def apply_night_vision(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    nv = np.zeros_like(frame); nv[:, :, 1] = gray
    return nv

def apply_sketch(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    inv = 255 - gray
    blur = cv2.GaussianBlur(inv, (21, 21), 0)
    sketch = cv2.divide(gray, 255 - blur, scale=256)
    return cv2.cvtColor(sketch, cv2.COLOR_GRAY2BGR)

def apply_pixelation(frame):
    h, w = frame.shape[:2]
    temp = cv2.resize(frame, (64, 64), interpolation=cv2.INTER_LINEAR)
    return cv2.resize(temp, (w, h), interpolation=cv2.INTER_NEAREST)

def apply_threshold(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    return cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)

def apply_retro_noise(frame):
    noise = np.zeros(frame.shape, np.uint8)
    cv2.randn(noise, (0,0,0), (50,50,50))
    return cv2.add(frame, noise)

def apply_oil_painting(frame):
    return cv2.stylization(frame, sigma_s=60, sigma_r=0.07)

# --- [새로 추가된 지능형 기능들] ---

# 얼굴 인식용 분류기 로드
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# 1. 움직임 감지용 변수
prev_frame = None

def func_motion_detector(frame):
    global prev_frame
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)
    
    if prev_frame is None:
        prev_frame = gray
        return frame
    
    # 프레임 차이 계산
    diff = cv2.absdiff(prev_frame, gray)
    _, thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
    prev_frame = gray
    
    # 차이가 있는 곳에 외곽선 그리기
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        if cv2.contourArea(c) < 500: continue
        (x, y, w, h) = cv2.boundingRect(c)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
    return frame

def func_face_detection(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(frame, 'Face', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
    return frame

def func_privacy_mask(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    for (x, y, w, h) in faces:
        face_roi = frame[y:y+h, x:x+w]
        # 얼굴 영역 모자이크 처리
        face_roi = cv2.resize(face_roi, (10, 10), interpolation=cv2.INTER_LINEAR)
        frame[y:y+h, x:x+w] = cv2.resize(face_roi, (w, h), interpolation=cv2.INTER_NEAREST)
    return frame

# 3. 사물 추적 (간단한 색상 기반 추적 예시: 초록색 대상)
def func_object_tracking(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    # 초록색 범위 (조정 가능)
    lower_green = np.array([35, 100, 100])
    upper_green = np.array([85, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        if cv2.contourArea(c) > 500:
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 3)
            cv2.putText(frame, "Tracking Green", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return frame

# --- [메인 실행 영역] ---

# 기능 매핑 (a, b, c, d 키 사용)
FUNCTIONS = {
    ord('a'): func_motion_detector,
    ord('s'): func_face_detection,
    ord('d'): func_object_tracking,
    ord('f'): func_privacy_mask
}

FILTERS = {
    ord('1'): apply_sepia, ord('2'): apply_cartoon, ord('3'): apply_night_vision,
    ord('4'): apply_sketch, ord('5'): apply_pixelation, ord('6'): apply_threshold,
    ord('7'): apply_retro_noise, ord('8'): apply_oil_painting
}

cap = cv2.VideoCapture(0)
active_mode = None
is_mirrored = True

print("="*50)
print(" [스마트 웹캠 통합 앱] ")
print(" 1~8: 필터 모드")
print(" a: 움직임 감지 | s: 얼굴 인식 | d: 색상 추적 | f: 모자이크")
print(" 0: 원본 | m: 거울 반전 | q: 종료")
print("="*50)

while True:
    ret, frame = cap.read()
    if not ret: break
    if is_mirrored: frame = cv2.flip(frame, 1)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): break
    elif key == ord('m'): is_mirrored = not is_mirrored
    elif key in FILTERS or key in FUNCTIONS: active_mode = key
    elif key == ord('0'): active_mode = None

    # 모드 적용
    mode_name = "ORIGINAL"
    if active_mode in FILTERS:
        frame = FILTERS[active_mode](frame)
        mode_name = FILTERS[active_mode].__name__.replace('apply_', '').upper()
    elif active_mode in FUNCTIONS:
        frame = FUNCTIONS[active_mode](frame)
        mode_name = FUNCTIONS[active_mode].__name__.replace('func_', '').upper()

    status = f"{mode_name} | MIRROR:{'ON' if is_mirrored else 'OFF'}"
    cv2.putText(frame, status, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.imshow('Smart WebCam Master', frame)

cap.release()
cv2.destroyAllWindows()

 [스마트 웹캠 통합 앱] 
 1~8: 필터 모드
 a: 움직임 감지 | s: 얼굴 인식 | d: 색상 추적 | f: 모자이크
 0: 원본 | m: 거울 반전 | q: 종료


In [12]:
import cv2
import numpy as np
import random

# --- [1~8번 기존 필터 함수들] ---
def apply_sepia(frame):
    sepia_matrix = np.array([[0.131, 0.534, 0.272], [0.168, 0.686, 0.349], [0.189, 0.769, 0.393]])
    return np.clip(cv2.transform(frame, sepia_matrix), 0, 255).astype(np.uint8)

def apply_cartoon(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)
    edges = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 9)
    color = cv2.bilateralFilter(frame, 9, 250, 250)
    return cv2.bitwise_and(color, color, mask=edges)

def apply_night_vision(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    nv = np.zeros_like(frame); nv[:, :, 1] = gray
    return nv

def apply_sketch(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    inv = 255 - gray
    blur = cv2.GaussianBlur(inv, (21, 21), 0)
    sketch = cv2.divide(gray, 255 - blur, scale=256)
    return cv2.cvtColor(sketch, cv2.COLOR_GRAY2BGR)

def apply_pixelation(frame):
    h, w = frame.shape[:2]
    temp = cv2.resize(frame, (64, 64), interpolation=cv2.INTER_LINEAR)
    return cv2.resize(temp, (w, h), interpolation=cv2.INTER_NEAREST)

# --- [게임용 전역 변수] ---
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
item_x = random.randint(50, 600)
item_y = 0
score = 0
speed = 15

def func_heading_game(frame):
    global item_x, item_y, score, speed
    
    h, w = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    # 아이템(빨간 공) 떨어뜨리기
    item_y += speed
    cv2.circle(frame, (item_x, item_y), 20, (0, 0, 255), -1)
    cv2.putText(frame, "CATCH!", (item_x - 20, item_y - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    
    # 충돌 감지 (얼굴 영역과 아이템 좌표 비교)
    for (x, y, fw, fh) in faces:
        cv2.rectangle(frame, (x, y), (x+fw, y+fh), (255, 255, 0), 2)
        
        # 아이템의 중심이 얼굴 사각형 안에 들어왔는지 체크
        if x < item_x < x+fw and y < item_y < y+fh:
            score += 1
            item_y = 0
            item_x = random.randint(50, w-50)
            speed += 1 # 잡을수록 조금씩 빨라짐
            # 점수 획득 이펙트 (잠시 하얗게 반짝)
            frame = cv2.add(frame, 50) 

    # 바닥에 놓친 경우
    if item_y > h:
        item_y = 0
        item_x = random.randint(50, w-50)
        score = max(0, score - 1) # 놓치면 감점
        speed = 15 # 속도 초기화

    # 화면에 점수 표시
    cv2.putText(frame, f"SCORE: {score}", (w-200, 50), cv2.FONT_HERSHEY_DUPLEX, 1.2, (255, 255, 255), 2)
    return frame

# --- [기타 지능형 기능들] ---
prev_frame = None
def func_motion_detector(frame):
    global prev_frame
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)
    if prev_frame is None: prev_frame = gray; return frame
    diff = cv2.absdiff(prev_frame, gray)
    _, thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
    prev_frame = gray
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        if cv2.contourArea(c) < 500: continue
        (x, y, w, h) = cv2.boundingRect(c)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
    return frame

def func_face_detection(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
    return frame

def func_privacy_mask(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    for (x, y, w, h) in faces:
        face_roi = frame[y:y+h, x:x+w]
        face_roi = cv2.resize(face_roi, (10, 10))
        frame[y:y+h, x:x+w] = cv2.resize(face_roi, (w, h), interpolation=cv2.INTER_NEAREST)
    return frame

# --- [메인 설정 및 루프] ---
FILTERS = {ord('1'): apply_sepia, ord('2'): apply_cartoon, ord('3'): apply_night_vision, ord('4'): apply_sketch, ord('5'): apply_pixelation}
FUNCTIONS = {ord('a'): func_motion_detector, ord('s'): func_face_detection, ord('f'): func_privacy_mask, ord('g'): func_heading_game}

cap = cv2.VideoCapture(0)
active_mode = None
is_mirrored = True

print("-" * 50)
print(" [OpenCV 스마트 웹캠 & 게임 앱] ")
print(" 1~5: 필터 모드")
print(" a: 움직임 감지 | s: 얼굴 인식 | f: 모자이크")
print(" g: 헤딩 게임 시작 (얼굴로 공 받기!)")
print(" 0: 원본 | m: 거울 반전 | q: 종료")
print("-" * 50)

while True:
    ret, frame = cap.read()
    if not ret: break
    if is_mirrored: frame = cv2.flip(frame, 1)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): break
    elif key == ord('m'): is_mirrored = not is_mirrored
    elif key in FILTERS or key in FUNCTIONS: active_mode = key
    elif key == ord('0'): active_mode = None

    # 모드 실행
    mode_name = "ORIGINAL"
    if active_mode in FILTERS:
        frame = FILTERS[active_mode](frame)
        mode_name = FILTERS[active_mode].__name__.upper()
    elif active_mode in FUNCTIONS:
        frame = FUNCTIONS[active_mode](frame)
        mode_name = FUNCTIONS[active_mode].__name__.upper()

    # 정보 출력
    cv2.putText(frame, f"MODE: {mode_name}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.imshow('OpenCV Game & Filter', frame)

cap.release()
cv2.destroyAllWindows()

--------------------------------------------------
 [OpenCV 스마트 웹캠 & 게임 앱] 
 1~5: 필터 모드
 a: 움직임 감지 | s: 얼굴 인식 | f: 모자이크
 g: 헤딩 게임 시작 (얼굴로 공 받기!)
 0: 원본 | m: 거울 반전 | q: 종료
--------------------------------------------------


In [13]:
import cv2
import numpy as np
import pygame
import os

# --- [사운드 설정] ---
pygame.mixer.init()

# 실제 드럼 사운드 파일 경로 (파일이 없다면 경고 문구가 뜹니다)
# .wav 파일을 준비해서 소스 코드와 같은 폴더에 넣어주세요.
def load_sound(file_name):
    if os.path.exists(file_name):
        return pygame.mixer.Sound(file_name)
    else:
        print(f"경고: {file_name} 파일을 찾을 수 없습니다. 무음으로 진행합니다.")
        return None

# 사운드 로드 (파일명은 본인의 파일에 맞게 수정하세요)
sound_snare = load_sound('snare.mp3')
sound_hihat = load_sound('hihat.mp3')
sound_bass = load_sound('bass.mp3')

# --- [드럼 객체 설정] ---
DRUMS = [
    {'name': 'SNARE', 'pos': (150, 350), 'radius': 60, 'color': (255, 255, 0), 'sound': sound_snare, 'active': False},
    {'name': 'HI-HAT', 'pos': (320, 150), 'radius': 50, 'color': (0, 255, 255), 'sound': sound_hihat, 'active': False},
    {'name': 'BASS', 'pos': (490, 350), 'radius': 70, 'color': (0, 255, 0), 'sound': sound_bass, 'active': False}
]

TOUCH_THRESHOLD = 0.18  # 감도 조절 (0.1 ~ 0.3)
cap = cv2.VideoCapture(0)
prev_gray = None
is_mirrored = True

print("-" * 50)
print(" [사운드 가상 드럼 시작] ")
print(" - 'm': 거울 모드 | 'q': 종료")
print("-" * 50)

while True:
    ret, frame = cap.read()
    if not ret: break
    
    if is_mirrored:
        frame = cv2.flip(frame, 1)
    
    display_frame = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)
    
    if prev_gray is None:
        prev_gray = gray
        continue

    # 움직임 추출
    frame_diff = cv2.absdiff(prev_gray, gray)
    _, thresh = cv2.threshold(frame_diff, 25, 255, cv2.THRESH_BINARY)
    prev_gray = gray

    for drum in DRUMS:
        x, y = drum['pos']
        r = drum['radius']
        
        # ROI 영역 내 움직임 계산
        mask = np.zeros_like(thresh)
        cv2.circle(mask, (x, y), r, 255, -1)
        roi = cv2.bitwise_and(thresh, mask)
        
        movement_ratio = np.count_nonzero(roi) / (np.pi * r**2)
        
        # --- [사운드 재생 핵심 로직] ---
        # 1. 움직임이 기준치 이상이고, 2. 이전 프레임에서 '미활성' 상태였을 때만 소리 재생 (연사 방지)
        if movement_ratio > TOUCH_THRESHOLD:
            if not drum['active']: 
                if drum['sound']:
                    drum['sound'].play()
                drum['active'] = True # 활성화 상태로 변경
            
            # 터치 시 시각 효과
            cv2.circle(display_frame, (x, y), r, (0, 0, 255), -1)
        else:
            drum['active'] = False # 손을 떼면 다시 활성화 가능 상태로
            cv2.circle(display_frame, (x, y), r, drum['color'], 2)

        cv2.putText(display_frame, drum['name'], (x-40, y+5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.imshow('OpenCV Virtual Drum (Sound ON)', display_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): break
    elif key == ord('m'): is_mirrored = not is_mirrored

cap.release()
cv2.destroyAllWindows()
pygame.quit()

ModuleNotFoundError: No module named 'pygame'

---
## Step 6 — Day2 감정/포즈 파이프라인 실시간 연결

Day2에서 만든 `analyze_image()` 를 웹캠 실시간 스트림에 연결합니다.

```
웹캠 프레임
    ├→ 얼굴 검출  →  감정 인식  →  결과 오버레이
    └→ 관절 추정  →  스틱맨
```

> ⚠️ 3개 모델을 매 프레임마다 실행하면 느릴 수 있어요.  
> N프레임마다 한 번씩 분석하는 방식으로 속도를 조절합니다.

omz_downloader --name face-detection-adas-0001 --output_dir ./models

omz_downloader --name emotions-recognition-retail-0003 --output_dir ./models

omz_downloader --name human-pose-estimation-0001 --output_dir ./models

In [14]:
import os
print("현재 작업 경로:", os.getcwd())
print("모델 파일 존재 여부:", os.path.exists('./models/intel/face-detection-adas-0001/FP32/face-detection-adas-0001.xml'))

현재 작업 경로: c:\Users\ComHolic\Downloads
모델 파일 존재 여부: False


In [15]:
from openvino.runtime import Core
import numpy as np

# Day2에서 만든 모델 로드 (로컬 경로)
MODEL_DIR = r'C:\Users\ComHolic\Desktop\AI_Folder\AI\04. Coding\VSCode\cv\models'
ie = Core()

face_compiled    = ie.compile_model(ie.read_model(f'{MODEL_DIR}/intel/face-detection-adas-0001/FP32/face-detection-adas-0001.xml'), 'CPU')
emotion_compiled = ie.compile_model(ie.read_model(f'{MODEL_DIR}/intel/emotions-recognition-retail-0003/FP32/emotions-recognition-retail-0003.xml'), 'CPU')
pose_compiled    = ie.compile_model(ie.read_model(f'{MODEL_DIR}/intel/human-pose-estimation-0001/FP32/human-pose-estimation-0001.xml'), 'CPU')

EMOTIONS   = ['neutral', 'happy', 'sad', 'surprise', 'anger']
EMOTION_COLOR = {'happy':(0,255,0),'neutral':(255,255,0),'sad':(255,100,0),'surprise':(0,200,255),'anger':(0,0,255)}
POSE_PAIRS = [(1,2),(1,5),(2,3),(3,4),(5,6),(6,7),(1,8),(8,9),(9,10),(1,11),(11,12),(12,13),(1,0),(0,14),(14,16),(0,15),(15,17)]

print('모델 로드 완료!')


모델 로드 완료!


In [16]:
from PIL import ImageFont, ImageDraw, Image
import numpy as np
import cv2

# 한글 폰트 설정
font_path = "C:/Windows/Fonts/malgun.ttf"
font_small  = ImageFont.truetype(font_path, 20)
font_medium = ImageFont.truetype(font_path, 26)

def put_korean_text(img, text, position, color=(0, 255, 255), font=None):
    if font is None:
        font = font_medium
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    draw    = ImageDraw.Draw(img_pil)
    r, g, b = color[2], color[1], color[0]
    draw.text(position, text, font=font, fill=(r, g, b))
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

# 실시간 감정 + 포즈 분석 앱
camera      = cv2.VideoCapture(0)
frame_skip  = 5
frame_count = 0
last_result = None

def analyze_frame(frame):
    result = frame.copy()
    h, w   = frame.shape[:2]

    # 1. 얼굴 검출
    fi      = face_compiled.input(0)
    fh, fw  = fi.shape[2], fi.shape[3]
    f_in    = np.expand_dims(cv2.resize(frame, (fw, fh)).transpose(2, 0, 1), 0)
    dets    = face_compiled([f_in])[face_compiled.output(0)]

    for det in dets[0][0]:
        if det[2] > 0.5:
            x1, y1 = max(0, int(det[3]*w)), max(0, int(det[4]*h))
            x2, y2 = min(w, int(det[5]*w)), min(h, int(det[6]*h))

            # 2. 감정 인식
            roi = frame[y1:y2, x1:x2]
            if roi.size > 0:
                e_in  = np.expand_dims(cv2.resize(roi, (64, 64)).transpose(2, 0, 1), 0).astype(np.float32)
                probs = emotion_compiled([e_in])[emotion_compiled.output(0)][0].flatten()
                emo   = EMOTIONS[int(np.argmax(probs))]
                color = EMOTION_COLOR.get(emo, (255, 255, 255))

                cv2.rectangle(result, (x1, y1), (x2, y2), color, 2)
                label_y = max(0, y1 - 35)
                result  = put_korean_text(result, f'{emo} {probs.max():.0%}', (x1, label_y), color, font_medium)

    # 3. 관절 포즈 추정
    pi      = pose_compiled.input(0)
    ph, pw  = pi.shape[2], pi.shape[3]
    p_in    = np.expand_dims(cv2.resize(frame, (pw, ph)).transpose(2, 0, 1), 0).astype(np.float32)
    heatmaps = pose_compiled([p_in])[pose_compiled.output(1)]

    keypoints = []
    for i in range(18):
        hm = heatmaps[0, i]
        _, conf, _, pt = cv2.minMaxLoc(hm)
        keypoints.append((int(pt[0]*w/hm.shape[1]), int(pt[1]*h/hm.shape[0]), float(conf)))

    for (p1, p2) in POSE_PAIRS:
        x1, y1, c1 = keypoints[p1]
        x2, y2, c2 = keypoints[p2]
        if c1 > 0.1 and c2 > 0.1:
            cv2.line(result, (x1, y1), (x2, y2), (0, 255, 255), 2)
    for (x, y, c) in keypoints:
        if c > 0.1:
            cv2.circle(result, (x, y), 4, (0, 200, 255), -1)

    return result

print('앱 시작!  q: 종료')
print(f'frame_skip={frame_skip} — 위/아래 방향키로 조절')

while True:
    ret, frame = camera.read()
    if not ret:
        break

    frame_count += 1
    if frame_count % frame_skip == 0:
        last_result = analyze_frame(frame)

    display = last_result if last_result is not None else frame.copy()

    # 상단 정보 오버레이 (한글)
    display = put_korean_text(display, f'분석 간격: {frame_skip}프레임 (+빠르게  -느리게)', (10, 10), (200, 200, 200), font_small)

    cv2.imshow('실시간 감정+포즈 분석 (q:종료)', display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == 45:    # + 키 → 더 자주 분석
        frame_skip = max(1, frame_skip - 1)
        print(f'frame_skip: {frame_skip}')
    elif key == 61:    # - 키 → 덜 자주 분석
        frame_skip = min(30, frame_skip + 1)
        print(f'frame_skip: {frame_skip}')

camera.release()
cv2.destroyAllWindows()

앱 시작!  q: 종료
frame_skip=5 — 위/아래 방향키로 조절


### 🎯 GenAI 활용 — 파이프라인 확장

---
**프롬프트 A: 감정별 색상 테두리 굵기 조절**
```
현재 감정 인식 앱에서 감정에 따라 바운딩 박스 테두리 굵기를 다르게 해줘.
happy → 굵기 4 (강조)
anger → 굵기 4 + 박스 깜빡임 효과 (프레임마다 색상 토글)
나머지 → 굵기 2
```

**프롬프트 B: 감정 히스토리 그래프**
```
실시간 웹캠 감정 인식 앱에서
최근 50프레임의 감정 분포를 화면 오른쪽에 막대 그래프로 실시간 표시해줘.
cv2로 직접 그리고, 각 감정(neutral/happy/sad/surprise/anger)마다
다른 색상 막대를 표시해줘.
```
---

In [17]:
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
# your code here
import cv2
import numpy as np
from PIL import ImageFont, ImageDraw, Image

# --- 1. 초기 설정 (폰트 및 더미 변수 - 실제 모델 환경에 맞게 수정 필요) ---
# 실제 환경에서는 openvino 또는 지원되는 라이브러리로 모델을 로드한 상태여야 합니다.
font_path = "C:/Windows/Fonts/malgun.ttf"
try:
    font_small  = ImageFont.truetype(font_path, 20)
    font_medium = ImageFont.truetype(font_path, 26)
except:
    font_small = ImageFont.load_default()
    font_medium = ImageFont.load_default()

# 예시용 감정 리스트 및 색상 (사용자 코드 기반)
EMOTIONS = ["neutral", "happy", "sad", "surprise", "anger"]
EMOTION_COLOR = {
    "happy": (0, 255, 255),  # 노란색
    "anger": (0, 0, 255),    # 빨간색
    "neutral": (0, 255, 0),  # 초록색
    "sad": (255, 0, 0),      # 파란색
    "surprise": (255, 255, 0)# 하늘색
}

# --- 2. 한글 출력 함수 ---
def put_korean_text(img, text, position, color=(0, 255, 255), font=None):
    if font is None:
        font = font_medium
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    draw    = ImageDraw.Draw(img_pil)
    # PIL은 RGB, OpenCV는 BGR이므로 색상 순서 변경
    r, g, b = color[2], color[1], color[0]
    draw.text(position, text, font=font, fill=(r, g, b))
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

# --- 3. 프레임 분석 함수 (핵심 로직 포함) ---
def analyze_frame(frame, current_count):
    result = frame.copy()
    h, w   = frame.shape[:2]

    # [주의] 아래 모델 관련 변수(face_compiled 등)는 상단에서 정의되어 있어야 합니다.
    # 1. 얼굴 검출
    fi      = face_compiled.input(0)
    fh, fw  = fi.shape[2], fi.shape[3]
    f_in    = np.expand_dims(cv2.resize(frame, (fw, fh)).transpose(2, 0, 1), 0)
    dets    = face_compiled([f_in])[face_compiled.output(0)]

    for det in dets[0][0]:
        if det[2] > 0.5:
            x1, y1 = max(0, int(det[3]*w)), max(0, int(det[4]*h))
            x2, y2 = min(w, int(det[5]*w)), min(h, int(det[6]*h))

            # 2. 감정 인식
            roi = frame[y1:y2, x1:x2]
            if roi.size > 0:
                e_in  = np.expand_dims(cv2.resize(roi, (64, 64)).transpose(2, 0, 1), 0).astype(np.float32)
                probs = emotion_compiled([e_in])[emotion_compiled.output(0)][0].flatten()
                emo   = EMOTIONS[int(np.argmax(probs))]
                
                # --- [감정별 스타일 로직 시작] ---
                base_color = EMOTION_COLOR.get(emo, (255, 255, 255))
                draw_color = base_color
                thickness = 2  # 기본 굵기

                if emo == "happy":
                    thickness = 4  # 굵기 강조
                
                elif emo == "anger":
                    thickness = 4  # 굵기 강조
                    # 3프레임 주기로 색상 토글 (깜빡임 효과)
                    # current_count를 활용해 실시간으로 색상 변경
                    if (current_count // 3) % 2 == 0:
                        draw_color = (0, 0, 255)   # 빨간색
                    else:
                        draw_color = (255, 255, 255) # 흰색 (깜빡임)
                # --- [감정별 스타일 로직 끝] ---

                # 바운딩 박스 그리기
                cv2.rectangle(result, (x1, y1), (x2, y2), draw_color, thickness)
                
                # 텍스트 레이블 출력
                label_y = max(0, y1 - 35)
                result  = put_korean_text(result, f'{emo} {probs.max():.0%}', (x1, label_y), draw_color, font_medium)

    # 3. 관절 포즈 추정 (기존 로직 유지)
    pi      = pose_compiled.input(0)
    ph, pw  = pi.shape[2], pi.shape[3]
    p_in    = np.expand_dims(cv2.resize(frame, (pw, ph)).transpose(2, 0, 1), 0).astype(np.float32)
    heatmaps = pose_compiled([p_in])[pose_compiled.output(1)]

    keypoints = []
    for i in range(18):
        hm = heatmaps[0, i]
        _, conf, _, pt = cv2.minMaxLoc(hm)
        keypoints.append((int(pt[0]*w/hm.shape[1]), int(pt[1]*h/hm.shape[0]), float(conf)))

    for (p1, p2) in POSE_PAIRS:
        x1, y1, c1 = keypoints[p1]
        x2, y2, c2 = keypoints[p2]
        if c1 > 0.1 and c2 > 0.1:
            cv2.line(result, (x1, y1), (x2, y2), (0, 255, 255), 2)
            
    for (x, y, c) in keypoints:
        if c > 0.1:
            cv2.circle(result, (x, y), 4, (0, 200, 255), -1)

    return result

# --- 4. 메인 루프 ---
camera      = cv2.VideoCapture(0)
frame_skip  = 5
frame_count = 0
last_result = None

print('앱 시작! q: 종료')
print(f'현재 분석 간격: {frame_skip} (위/아래 방향키로 조절)')

while True:
    ret, frame = camera.read()
    if not ret:
        break

    frame_count += 1
    
    # 분석 주기(frame_skip)마다 분석을 수행하지만, 
    # 분노 깜빡임 효과를 위해 frame_count를 전달합니다.
    if frame_count % frame_skip == 0:
        last_result = analyze_frame(frame, frame_count)

    display = last_result if last_result is not None else frame.copy()

    # 상단 정보 오버레이
    display = put_korean_text(display, f'분석 간격: {frame_skip}프레임 (+/- 키로 조절)', (10, 10), (200, 200, 200), font_small)

    cv2.imshow('실시간 감정+포즈 분석 (q:종료)', display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('=') or key == ord('+'): # + 키 (ASCII 61, 43)
        frame_skip = max(1, frame_skip - 1)
        print(f'frame_skip: {frame_skip}')
    elif key == ord('-'): # - 키 (ASCII 45)
        frame_skip = min(30, frame_skip + 1)
        print(f'frame_skip: {frame_skip}')

camera.release()
cv2.destroyAllWindows()

앱 시작! q: 종료
현재 분석 간격: 5 (위/아래 방향키로 조절)


---
## 🎯 도전 과제 3 — Streamlit으로 웹 앱 배포하기

지금까지 만든 필터 앱은 **내 컴퓨터에서만** 실행됩니다.  
Streamlit을 사용하면 누구나 브라우저로 접속할 수 있는 **웹 앱**으로 만들 수 있습니다!

### Streamlit이란?
Python 코드만으로 웹 앱을 만들 수 있는 라이브러리입니다.  
HTML/CSS/JavaScript를 전혀 몰라도 됩니다.

```
지금까지        →       Streamlit 적용 후
cv2.imshow()          브라우저에서 실시간 확인
로컬에서만 실행        링크 공유로 누구나 접속
키보드로 조작          버튼/슬라이더로 조작
```

### 동작 방식 (웹캠 대신 이미지 업로드)
웹 환경에서는 웹캠 직접 접근이 제한되므로  
**이미지를 업로드하면 필터를 적용해서 보여주는 앱**으로 만듭니다.

```
이미지 업로드  ->  필터 선택 (사이드바)  ->  원본 vs 결과 나란히 표시
```

### Step 1 — Streamlit 설치 확인

아래 셀을 실행해서 설치 여부를 확인하세요.

In [53]:
try:
    import streamlit
    print("Streamlit 버전:", streamlit.__version__)
except ImportError:
    print("미설치. 아래 명령어로 설치하세요:")
    print("  pip install streamlit")

미설치. 아래 명령어로 설치하세요:
  pip install streamlit


### Step 2 — GenAI로 앱 코드 생성하기

아래 프롬프트를 그대로 복사해서 GenAI에게 전달하세요.  
받은 코드를 `filter_app.py` 파일로 저장합니다.

---
**프롬프트:**
```
Python Streamlit과 OpenCV로 이미지 필터 비교 웹 앱을 만들어줘.

[앱 구조]
- 사이드바에서 필터 선택 (selectbox)
- 메인 화면에 이미지 업로드 (file_uploader, jpg/png)
- 업로드한 이미지에 선택한 필터 적용
- 원본과 필터 적용 결과를 나란히(columns) 표시
- 필터 적용된 이미지 다운로드 버튼 추가

[필터 목록] — 각각 함수로 구현
1. 원본 (변환 없음)
2. 회색조 (Grayscale)
3. Gaussian 블러 (커널 크기를 슬라이더로 조정, 범위 1~31 홀수만)
4. Canny 엣지 (threshold1, threshold2를 슬라이더로 조정)
5. 세피아 (numpy 행렬 변환)
6. 선명화 (Sharpening kernel)

[기술 조건]
- OpenCV는 BGR이므로 PIL 또는 cvtColor로 RGB 변환 후 st.image() 사용
- 업로드 이미지는 np.frombuffer + cv2.imdecode로 읽기
- 다운로드 버튼은 st.download_button 사용, PNG 형식
- 파일명: filter_app.py
- 한국어 UI (제목, 레이블, 설명 모두 한국어)

[앱 제목]: "이미지 필터 비교 앱"
[사이드바 제목]: "필터 설정"
```
---

In [ ]:
import streamlit as st
import cv2
import numpy as np
from PIL import Image
import io

# --- [페이지 설정] ---
st.set_page_config(page_title="이미지 필터 비교 앱", layout="wide")

st.title("📸 이미지 필터 비교 앱")
st.markdown("이미지를 업로드하고 다양한 필터를 적용해 보세요. 결과물을 바로 다운로드할 수 있습니다.")

# --- [필터 함수 정의] ---

def apply_grayscale(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def apply_blur(img, kernel_size):
    # GaussianBlur의 커널 크기는 홀수여야 함
    return cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)

def apply_canny(img, t1, t2):
    return cv2.Canny(img, t1, t2)

def apply_sepia(img):
    # 세피아 변환 행렬
    img_sepia = np.array(img, dtype=np.float64)
    img_sepia = cv2.transform(img_sepia, np.matrix([
        [0.272, 0.534, 0.131],
        [0.349, 0.686, 0.168],
        [0.393, 0.769, 0.189]
    ]))
    img_sepia = np.clip(img_sepia, 0, 255)
    return img_sepia.astype(np.uint8)

def apply_sharpening(img):
    kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])
    return cv2.filter2D(img, -1, kernel)

# --- [사이드바 필터 설정] ---
st.sidebar.header("⚙️ 필터 설정")
filter_option = st.sidebar.selectbox(
    "적용할 필터를 선택하세요",
    ["원본", "회색조", "Gaussian 블러", "Canny 엣지", "세피아", "선명화"]
)

# 필터별 상세 옵션 (사이드바)
params = {}
if filter_option == "Gaussian 블러":
    params['kernel'] = st.sidebar.slider("블러 강도 (커널 크기)", 1, 31, 15, step=2)
elif filter_option == "Canny 엣지":
    params['t1'] = st.sidebar.slider("Threshold 1", 0, 255, 100)
    params['t2'] = st.sidebar.slider("Threshold 2", 0, 255, 200)

# --- [메인 기능: 이미지 업로드] ---
uploaded_file = st.file_uploader("이미지 파일을 업로드하세요 (JPG, PNG)", type=['jpg', 'png', 'jpeg'])

if uploaded_file is not None:
    # 이미지 읽기
    file_bytes = np.frombuffer(uploaded_file.read(), np.uint8)
    img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    
    # 원본 복사 (출력용 RGB 변환)
    original_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # 필터 적용 로직
    result_img = img.copy()
    
    if filter_option == "회색조":
        result_img = apply_grayscale(img)
    elif filter_option == "Gaussian 블러":
        result_img = apply_blur(img, params['kernel'])
    elif filter_option == "Canny 엣지":
        result_img = apply_canny(img, params['t1'], params['t2'])
    elif filter_option == "세피아":
        result_img = apply_sepia(img)
    elif filter_option == "선명화":
        result_img = apply_sharpening(img)
    
    # OpenCV 결과물을 다시 RGB로 변환 (st.image 출력용)
    if len(result_img.shape) == 2:  # Gray or Canny
        result_rgb = cv2.cvtColor(result_img, cv2.COLOR_GRAY2RGB)
    else:
        result_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

    # --- [원본 vs 결과 나란히 표시] ---
    col1, col2 = st.columns(2)
    
    with col1:
        st.subheader("원본 이미지")
        st.image(original_rgb, use_container_width=True)
        
    with col2:
        st.subheader(f"필터 적용: {filter_option}")
        st.image(result_rgb, use_container_width=True)

    # --- [다운로드 버튼] ---
    # 결과 이미지를 PNG 바이트로 변환
    is_success, buffer = cv2.imencode(".png", result_img)
    if is_success:
        io_buf = io.BytesIO(buffer)
        st.sidebar.markdown("---")
        st.sidebar.download_button(
            label="🖼️ 결과 이미지 다운로드",
            data=io_buf.getvalue(),
            file_name=f"filtered_{filter_option}.png",
            mime="image/png"
        )

else:
    st.info("이미지를 업로드하면 필터 효과를 확인할 수 있습니다.")

In [54]:
# 받은 코드를 filter_app.py로 저장하는 방법

app_code = """
import streamlit as st
import cv2
import numpy as np
from PIL import Image
import io

# --- [페이지 설정] ---
st.set_page_config(page_title="이미지 필터 비교 앱", layout="wide")

st.title("📸 이미지 필터 비교 앱")
st.markdown("이미지를 업로드하고 다양한 필터를 적용해 보세요. 결과물을 바로 다운로드할 수 있습니다.")

# --- [필터 함수 정의] ---

def apply_grayscale(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def apply_blur(img, kernel_size):
    # GaussianBlur의 커널 크기는 홀수여야 함
    return cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)

def apply_canny(img, t1, t2):
    return cv2.Canny(img, t1, t2)

def apply_sepia(img):
    # 세피아 변환 행렬
    img_sepia = np.array(img, dtype=np.float64)
    img_sepia = cv2.transform(img_sepia, np.matrix([
        [0.272, 0.534, 0.131],
        [0.349, 0.686, 0.168],
        [0.393, 0.769, 0.189]
    ]))
    img_sepia = np.clip(img_sepia, 0, 255)
    return img_sepia.astype(np.uint8)

def apply_sharpening(img):
    kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])
    return cv2.filter2D(img, -1, kernel)

# --- [사이드바 필터 설정] ---
st.sidebar.header("⚙️ 필터 설정")
filter_option = st.sidebar.selectbox(
    "적용할 필터를 선택하세요",
    ["원본", "회색조", "Gaussian 블러", "Canny 엣지", "세피아", "선명화"]
)

# 필터별 상세 옵션 (사이드바)
params = {}
if filter_option == "Gaussian 블러":
    params['kernel'] = st.sidebar.slider("블러 강도 (커널 크기)", 1, 31, 15, step=2)
elif filter_option == "Canny 엣지":
    params['t1'] = st.sidebar.slider("Threshold 1", 0, 255, 100)
    params['t2'] = st.sidebar.slider("Threshold 2", 0, 255, 200)

# --- [메인 기능: 이미지 업로드] ---
uploaded_file = st.file_uploader("이미지 파일을 업로드하세요 (JPG, PNG)", type=['jpg', 'png', 'jpeg'])

if uploaded_file is not None:
    # 이미지 읽기
    file_bytes = np.frombuffer(uploaded_file.read(), np.uint8)
    img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    
    # 원본 복사 (출력용 RGB 변환)
    original_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # 필터 적용 로직
    result_img = img.copy()
    
    if filter_option == "회색조":
        result_img = apply_grayscale(img)
    elif filter_option == "Gaussian 블러":
        result_img = apply_blur(img, params['kernel'])
    elif filter_option == "Canny 엣지":
        result_img = apply_canny(img, params['t1'], params['t2'])
    elif filter_option == "세피아":
        result_img = apply_sepia(img)
    elif filter_option == "선명화":
        result_img = apply_sharpening(img)
    
    # OpenCV 결과물을 다시 RGB로 변환 (st.image 출력용)
    if len(result_img.shape) == 2:  # Gray or Canny
        result_rgb = cv2.cvtColor(result_img, cv2.COLOR_GRAY2RGB)
    else:
        result_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

    # --- [원본 vs 결과 나란히 표시] ---
    col1, col2 = st.columns(2)
    
    with col1:
        st.subheader("원본 이미지")
        st.image(original_rgb, use_container_width=True)
        
    with col2:
        st.subheader(f"필터 적용: {filter_option}")
        st.image(result_rgb, use_container_width=True)

    # --- [다운로드 버튼] ---
    # 결과 이미지를 PNG 바이트로 변환
    is_success, buffer = cv2.imencode(".png", result_img)
    if is_success:
        io_buf = io.BytesIO(buffer)
        st.sidebar.markdown("---")
        st.sidebar.download_button(
            label="🖼️ 결과 이미지 다운로드",
            data=io_buf.getvalue(),
            file_name=f"filtered_{filter_option}.png",
            mime="image/png"
        )

else:
    st.info("이미지를 업로드하면 필터 효과를 확인할 수 있습니다.")
"""

with open("filter_app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("filter_app.py 저장 완료!")

filter_app.py 저장 완료!


### Step 3 — 로컬에서 실행하기

파일을 저장했으면 **터미널**에서 아래 명령어를 실행하세요.

```bash
streamlit run filter_app.py
```

브라우저가 자동으로 열리면서 `http://localhost:8501` 로 접속됩니다.

### Step 4 — Streamlit Cloud로 배포하기

로컬에서 잘 동작하면 인터넷에 배포해서 링크를 공유할 수 있습니다.

**배포 순서**

1. **GitHub에 코드 올리기**
```bash
# 터미널에서
git init
git add filter_app.py
git commit -m "Add filter app"
git push origin main
```

2. **requirements.txt 만들기**  
아래 프롬프트로 GenAI에게 요청하세요.

---
**프롬프트:**
```
Streamlit과 OpenCV를 사용하는 Python 앱의 requirements.txt를 만들어줘.
필요한 패키지: streamlit, opencv-python-headless, numpy, Pillow
각 패키지의 최신 안정 버전을 포함해줘.

주의: 서버 환경에서는 opencv-python 대신
opencv-python-headless를 사용해야 해.
```
---

3. **Streamlit Cloud에서 배포**
   - [share.streamlit.io](https://share.streamlit.io) 접속
   - GitHub 계정으로 로그인
   - `New app` → 저장소 선택 → `filter_app.py` 선택 → `Deploy`
   - 몇 분 후 `https://[이름].streamlit.app` 링크 생성!

### 🎯 추가 도전 — 슬라이더로 HSV 마스크 실시간 조정

---
**프롬프트:**
```
기존 filter_app.py에 새로운 필터를 추가해줘.

[추가할 필터]: "HSV 색상 마스크"
- 사이드바에 H(색조), S(채도), V(명도) 각각의 최솟값/최댓값 슬라이더 6개 추가
  - H: 0~180, S: 0~255, V: 0~255
- 슬라이더 값으로 cv2.inRange() 마스크를 만들어 원본에 적용
- 마스크 이미지와 결과 이미지를 함께 표시
- 슬라이더를 움직이면 실시간으로 결과가 바뀌어야 함

기존 필터 목록에 "HSV 색상 마스크"를 추가하고
선택했을 때만 슬라이더가 사이드바에 나타나도록 해줘.
```
---

In [56]:
# 받은 코드를 filter_app.py로 저장하는 방법

app_code = """
import streamlit as st
import cv2
import numpy as np
from PIL import Image
import io

# --- [페이지 설정] ---
st.set_page_config(page_title="이미지 필터 비교 앱", layout="wide")

st.title("📸 이미지 필터 비교 앱")
st.markdown("이미지를 업로드하고 다양한 필터를 적용해 보세요. 결과물을 바로 다운로드할 수 있습니다.")

# --- [필터 함수 정의] ---

def apply_grayscale(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def apply_blur(img, kernel_size):
    return cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)

def apply_canny(img, t1, t2):
    return cv2.Canny(img, t1, t2)

def apply_sepia(img):
    img_sepia = np.array(img, dtype=np.float64)
    img_sepia = cv2.transform(img_sepia, np.matrix([
        [0.272, 0.534, 0.131],
        [0.349, 0.686, 0.168],
        [0.393, 0.769, 0.189]
    ]))
    img_sepia = np.clip(img_sepia, 0, 255)
    return img_sepia.astype(np.uint8)

def apply_sharpening(img):
    kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])
    return cv2.filter2D(img, -1, kernel)

# --- [신규: HSV 마스크 함수] ---
def apply_hsv_mask(img, h_range, s_range, v_range):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower = np.array([h_range[0], s_range[0], v_range[0]])
    upper = np.array([h_range[1], s_range[1], v_range[1]])
    mask = cv2.inRange(hsv, lower, upper)
    result = cv2.bitwise_and(img, img, mask=mask)
    return mask, result

# --- [사이드바 필터 설정] ---
st.sidebar.header("⚙️ 필터 설정")
filter_option = st.sidebar.selectbox(
    "적용할 필터를 선택하세요",
    ["원본", "회색조", "Gaussian 블러", "Canny 엣지", "세피아", "선명화", "HSV 색상 마스크"]
)

# 필터별 상세 옵션
params = {}
if filter_option == "Gaussian 블러":
    params['kernel'] = st.sidebar.slider("블러 강도 (커널 크기)", 1, 31, 15, step=2)
elif filter_option == "Canny 엣지":
    params['t1'] = st.sidebar.slider("Threshold 1", 0, 255, 100)
    params['t2'] = st.sidebar.slider("Threshold 2", 0, 255, 200)
elif filter_option == "HSV 색상 마스크":
    st.sidebar.subheader("HSV 범위 설정")
    params['h'] = st.sidebar.slider("H (색조) 범위", 0, 180, (0, 180))
    params['s'] = st.sidebar.slider("S (채도) 범위", 0, 255, (0, 255))
    params['v'] = st.sidebar.slider("V (명도) 범위", 0, 255, (0, 255))

# --- [메인 기능: 이미지 업로드] ---
uploaded_file = st.file_uploader("이미지 파일을 업로드하세요 (JPG, PNG)", type=['jpg', 'png', 'jpeg'])

if uploaded_file is not None:
    file_bytes = np.frombuffer(uploaded_file.read(), np.uint8)
    img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    original_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    result_img = img.copy()
    mask_img = None # HSV 마스크용
    
    if filter_option == "회색조":
        result_img = apply_grayscale(img)
    elif filter_option == "Gaussian 블러":
        result_img = apply_blur(img, params['kernel'])
    elif filter_option == "Canny 엣지":
        result_img = apply_canny(img, params['t1'], params['t2'])
    elif filter_option == "세피아":
        result_img = apply_sepia(img)
    elif filter_option == "선명화":
        result_img = apply_sharpening(img)
    elif filter_option == "HSV 색상 마스크":
        mask_img, result_img = apply_hsv_mask(img, params['h'], params['s'], params['v'])
    
    # 출력용 변환
    if len(result_img.shape) == 2:
        result_rgb = cv2.cvtColor(result_img, cv2.COLOR_GRAY2RGB)
    else:
        result_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

    # --- [화면 표시] ---
    if filter_option == "HSV 색상 마스크":
        # HSV 마스크일 때는 3개 열(원본, 마스크, 결과) 표시
        col1, col2, col3 = st.columns(3)
        with col1:
            st.subheader("원본")
            st.image(original_rgb, use_container_width=True)
        with col2:
            st.subheader("이진 마스크 (흑백)")
            st.image(mask_img, use_container_width=True)
        with col3:
            st.subheader("필터 결과")
            st.image(result_rgb, use_container_width=True)
    else:
        col1, col2 = st.columns(2)
        with col1:
            st.subheader("원본 이미지")
            st.image(original_rgb, use_container_width=True)
        with col2:
            st.subheader(f"필터 적용: {filter_option}")
            st.image(result_rgb, use_container_width=True)

    # --- [다운로드 버튼] ---
    is_success, buffer = cv2.imencode(".png", result_img)
    if is_success:
        st.sidebar.markdown("---")
        st.sidebar.download_button(
            label="🖼️ 결과 이미지 다운로드",
            data=io.BytesIO(buffer).getvalue(),
            file_name=f"filtered_{filter_option}.png",
            mime="image/png"
        )
else:
    st.info("이미지를 업로드하면 필터 효과를 확인할 수 있습니다.")
"""

with open("filter_app2.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("filter_app.py 저장 완료!")

filter_app.py 저장 완료!


---
## 오늘 완성한 것

### 오전
✅ 실시간 웹캠 필터 앱 (6가지 모드)
✅ 트랙바로 Canny 파라미터 실시간 조절
✅ 웹캠 영상 녹화 기능 (r키 토글)

### 오후
✅ Day2 감정/포즈 파이프라인 실시간 연결
✅ frame_skip으로 속도 조절
✅ Streamlit 웹 앱 배포

```
1~2일차 기초   →   3일차 통합
픽셀/마스킹        실시간 필터 앱
Edge/Contour       트랙바 파라미터 조절
OpenVINO 모델      감정+포즈 실시간 분석
```

> 내일은 OpenVINO로 **Streamlit 미니 프로젝트** 완성! 🎯